# Mini-MoE-CoT: Interactive Learning Notebook

This notebook walks through each concept in the pipeline interactively.
Run cells in order. No GPU required for the first 4 sections.

## Contents
1. **MoE Routing** — How the router decides which experts to use
2. **Expert Specialization** — Why experts diverge over training
3. **Chain-of-Thought Format** — How `<think>` traces are structured
4. **Tool Use Pattern** — The ReAct loop: Think → Act → Observe
5. **Distillation** — Calling the Ollama teacher (requires Ollama)
6. **Training Loop** — Mini training run on toy data (requires GPU)


In [ ]:
# Setup — run this first
import sys
sys.path.insert(0, '..')
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

---
## Part 1: MoE Routing

The Router is a simple linear layer: `hidden_size → num_experts`.
For each token, it outputs a score per expert. We take the top-k.


In [ ]:
from src.moe_layer import Router, Expert, MoELayer

# Create a small router for illustration
router = Router(hidden_size=16, num_experts=4, top_k=2)

# Simulate a batch of 5 tokens
tokens = torch.randn(1, 5, 16)  # (batch=1, seq=5, hidden=16)

weights, indices, logits = router(tokens)

print('For 5 tokens, top-2 expert assignments:')
for t in range(5):
    e1, e2 = indices[0, t, 0].item(), indices[0, t, 1].item()
    w1, w2 = weights[0, t, 0].item(), weights[0, t, 1].item()
    print(f'  Token {t}: Expert {e1} ({w1:.2f}) + Expert {e2} ({w2:.2f})')

print(f'\nWeights sum to 1: {weights[0].sum(dim=-1).tolist()}')

In [ ]:
# Visualize routing distribution
from src.moe_layer import compute_aux_loss

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scenario 1: Balanced routing (what we want)
balanced_logits = torch.randn(100, 4) * 0.1  # nearly uniform
balanced_probs = F.softmax(balanced_logits, dim=-1).mean(dim=0)
axes[0].bar(range(4), balanced_probs.detach().numpy(), color='steelblue')
axes[0].set_title('Balanced Routing\n(low aux loss)')
axes[0].set_xlabel('Expert ID')
axes[0].set_ylabel('Mean Routing Probability')
axes[0].axhline(0.25, color='red', linestyle='--', label='Ideal (0.25)')
axes[0].legend()
loss1 = compute_aux_loss(balanced_logits, 4, 2, 1.0)
axes[0].set_title(f'Balanced Routing\nAux Loss = {loss1.item():.3f}')

# Scenario 2: Collapsed routing (expert 0 gets everything)
collapsed_logits = torch.zeros(100, 4)
collapsed_logits[:, 0] = 5.0
collapsed_probs = F.softmax(collapsed_logits, dim=-1).mean(dim=0)
axes[1].bar(range(4), collapsed_probs.detach().numpy(), color='salmon')
axes[1].axhline(0.25, color='red', linestyle='--', label='Ideal (0.25)')
axes[1].legend()
loss2 = compute_aux_loss(collapsed_logits, 4, 2, 1.0)
axes[1].set_title(f'Collapsed Routing\nAux Loss = {loss2.item():.3f}')
axes[1].set_xlabel('Expert ID')

plt.tight_layout()
plt.savefig('routing_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Aux loss penalizes collapse: {loss1.item():.3f} (balanced) vs {loss2.item():.3f} (collapsed)')

---
## Part 2: Full MoE Forward Pass

Now let's run the complete MoE layer and verify shapes + gradients.


In [ ]:
# Full MoE layer test
moe = MoELayer(
    hidden_size=256,
    intermediate_size=512,
    num_experts=4,
    top_k=2,
    aux_loss_coef=0.01,
)

# Simulate a transformer hidden state
x = torch.randn(2, 32, 256, requires_grad=True)  # (batch=2, seq=32, hidden=256)

output, aux_loss = moe(x)

print(f'Input shape:  {tuple(x.shape)}')
print(f'Output shape: {tuple(output.shape)}')
print(f'Aux loss:     {aux_loss.item():.6f}')
print(f'Output != Input (MoE transformed): {not torch.allclose(x, output)}')

# Verify gradients flow back
total_loss = output.sum() + aux_loss
total_loss.backward()
print(f'Gradient on input: {x.grad.abs().mean().item():.6f} (nonzero = good)')

# Count active parameters
total_params = sum(p.numel() for p in moe.parameters())
active_params = sum(p.numel() for p in list(moe.experts.children())[:2])  # top-2 experts
print(f'\nTotal MoE params: {total_params:,}')
print(f'Active per token (top-2/4): ~{active_params:,} ({100*active_params/total_params:.0f}% of total)')

---
## Part 3: Chain-of-Thought Format

The training data uses a structured format with XML-like tags.
Let's explore what good vs. bad CoT looks like.


In [ ]:
from src.tool_loop import extract_answer, extract_thinking, ToolDispatcher

# Example of well-formed CoT + tool use output
good_response = """
<think>
The question asks for 15% of 847.
I'll use the calculator to be precise.
15% = 15/100 = 0.15
So I need: 0.15 * 847
</think>
<tool>calc(0.15 * 847)</tool>
<tool_result>Calculator result: 127.05</tool_result>
<think>
The calculator confirms: 15% of 847 = 127.05
Now I need to compare to sqrt(100) = 10.
127.05 > 10, so yes, it is more.
</think>
<answer>15% of 847 is 127.05, which is greater than √100 = 10.</answer>
"""

thinking = extract_thinking(good_response)
answer = extract_answer(good_response)

print('=== Extracted Thinking ===')
print(thinking)
print('\n=== Extracted Answer ===')
print(answer)

# Parse tool calls
dispatcher = ToolDispatcher()
calls = dispatcher.parse_tool_calls(good_response)
print(f'\n=== Tool Calls Found ===')
for name, args in calls:
    print(f'  {name}({args})')

---
## Part 4: Tool Use Execution Loop

The ToolDispatcher parses model output and executes tool calls.
Let's test each available tool.


In [ ]:
from tools.calculator import run as calc
from tools.search import run as search
from tools.datetime_tool import run as dt

# Calculator tests
print('=== Calculator Tool ===')
expressions = [
    '15 * 0.20',
    'sqrt(144)',
    '2 ** 10',
    '(100 + 200) / 3',
    'log(2.718281828)',  # ln(e) ≈ 1
]
for expr in expressions:
    print(f'  calc({expr}) → {calc(expr)}')

print('\n=== Search Tool ===')
queries = ['population Switzerland', 'speed of light', 'moon distance']
for q in queries:
    result = search(q)
    print(f'  search({q!r})\n    → {result[:70]}...')

print('\n=== Datetime Tool ===')
date_queries = ['today', 'year', 'days_until(2025-12-25)']
for q in date_queries:
    print(f'  datetime({q!r}) → {dt(q)}')

In [ ]:
# Simulate the full ReAct loop manually (without model)
print('=== Simulated ReAct Loop ===')

dispatcher = ToolDispatcher()

# Round 0: Model generates thinking + tool call
round0 = '''
<think>I need to find Switzerland's population, then compute 0.01% of it.</think>
<tool>search(population Switzerland)</tool>
'''
print(f'Round 0 (model output):\n{round0.strip()}')

# Dispatcher executes tool and injects result
enriched = dispatcher.inject_results(round0)
print(f'\nAfter tool execution:\n{enriched.strip()}')

# Round 1: Model sees result, calculates
round1 = '''
<think>Switzerland has ~8.9 million people. 
0.01% of 8,900,000 = 8,900,000 * 0.0001 = 890</think>
<tool>calc(8900000 * 0.0001)</tool>
'''
enriched1 = dispatcher.inject_results(round1)
print(f'\nRound 1 (with calc):\n{enriched1.strip()}')

# Round 2: Final answer
round2 = '<answer>0.01% of Switzerland\'s population (~8.9M) is approximately 890 people.</answer>'
print(f'\nFinal: {extract_answer(round2)}')
print(f'\nTotal tool calls made: {len(dispatcher.call_history)}')

---
## Part 5: Distillation — Calling the Teacher (requires Ollama)

This section requires Ollama running with a model pulled.
Start it with: `ollama run qwen3.5:7b`


In [ ]:
# Check Ollama availability
try:
    import ollama
    client = ollama.Client()
    models = client.list()
    print('Ollama available!')
    print('Models:', [m.model for m in models.models])
    OLLAMA_AVAILABLE = True
except Exception as e:
    print(f'Ollama not available: {e}')
    print('Start with: ollama run qwen3.5:7b')
    OLLAMA_AVAILABLE = False

In [ ]:
if OLLAMA_AVAILABLE:
    from src.distill import generate_with_teacher, build_teacher_prompt, validate_response
    
    # Pick first available model
    teacher_model = [m.model for m in client.list().models][0]
    print(f'Using teacher: {teacher_model}')
    
    # Generate one example
    question = 'A rectangle has length 3x width and perimeter 56cm. Find the area.'
    response = generate_with_teacher(
        question=question,
        include_tools=False,
        teacher_model=teacher_model,
        temperature=0.7,
        max_tokens=512,
    )
    
    print(f'\n=== Teacher Response ===')
    print(response)
    print(f'\n=== Validation ===')
    print(f'Valid: {validate_response(response, include_tools=False)}')
    print(f'Answer: {extract_answer(response)}')
else:
    print('Skipping (Ollama not available)')

---
## Part 6: Expert Specialization Over Training

This visualization shows what happens to routing distributions as training progresses.
We simulate it here; run `analyze_routing.py` after real training.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simulate expert utilization at different training stages
categories = ['Math', 'Reasoning', 'Tool Planning', 'Output']
num_experts = 4

# Before training: random routing (uniform)
before = np.array([
    [0.26, 0.25, 0.24, 0.25],  # Math
    [0.25, 0.25, 0.25, 0.25],  # Reasoning
    [0.24, 0.26, 0.25, 0.25],  # Tool Planning
    [0.25, 0.25, 0.25, 0.25],  # Output
])

# After training: specialized routing (emergent)
after = np.array([
    [0.55, 0.15, 0.20, 0.10],  # Math → Expert 0 specializes
    [0.15, 0.52, 0.18, 0.15],  # Reasoning → Expert 1 specializes
    [0.12, 0.18, 0.58, 0.12],  # Tool Planning → Expert 2 specializes
    [0.10, 0.15, 0.18, 0.57],  # Output → Expert 3 specializes
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, title in zip(axes, [before, after], ['Before Training', 'After Training']):
    im = ax.imshow(data, cmap='Blues', vmin=0, vmax=0.6, aspect='auto')
    ax.set_xticks(range(num_experts))
    ax.set_xticklabels([f'Expert {i}' for i in range(num_experts)])
    ax.set_yticks(range(len(categories)))
    ax.set_yticklabels(categories)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Expert')
    
    for i in range(len(categories)):
        for j in range(num_experts):
            ax.text(j, i, f'{data[i,j]:.2f}', ha='center', va='center',
                    color='white' if data[i,j] > 0.35 else 'black', fontsize=11)

plt.colorbar(im, ax=axes[1], label='Routing Probability')
plt.suptitle('Expert Specialization: MoE Routing Heatmap', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('expert_specialization.png', dpi=100, bbox_inches='tight')
plt.show()

print('Key insight: After training, each expert specializes in different token types.')
print('This emerges from the training signal — no explicit expert assignment!')

---
## Summary: What You've Learned

| Component | Key Insight |
|-----------|-------------|
| **MoE Router** | Softmax over N experts → top-k selection. Differentiable. |
| **Aux Loss** | Prevents collapse by penalizing uneven expert usage. |
| **CoT Format** | `<think>` traces are supervised: student imitates teacher reasoning. |
| **Tool Loop** | ReAct pattern: generate → parse tags → execute → inject → repeat. |
| **Distillation** | Sequence-level KD: train student on teacher's output tokens. |
| **Specialization** | Emergent behavior: experts develop distinct roles during training. |

### Connection to Production Systems (Swisscom context)

- **Controlled autonomy**: The `max_tool_rounds` limit in `ToolUseInferenceLoop` is a simple governance mechanism — the same principle scales to enterprise policy engines.
- **Observability**: The `call_history` in `ToolDispatcher` gives a full audit trail of every tool call — essential for regulated environments.
- **Expert routing as policy**: In a real system, the MoE router's specialization maps to domain-specific sub-agents (financial reasoning expert, legal expert, etc.).
